# MPP-IA — Pipeline complet

**Ordre d'exécution (à respecter la première fois) :**
1. Setup (Drive + clone repo + deps)
2. Kaggle : download des CSV bruts
3. **Vérifier les colonnes réelles** (`inspect_columns`) avant de lancer le feature engineering
4. Build dataset + split
5. Lancer le NAS (celui qu'on lance à 22h et qu'on reprend le lendemain)
6. Entraînement final / ensemble
7. Prédiction sur un match
8. Réentraînement incrémental (à relancer après chaque journée)

**Pour la nuit complète :** exécute les cellules 1 à 5, laisse tourner, ferme l'onglet si tu veux
mais garde la session Colab ouverte (le keep-alive en cellule 5bis aide à éviter le timeout).

## 1. Setup — Drive, clone du repo, dépendances

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Dossier persistant sur Drive pour les checkpoints (survit à une déconnexion Colab)
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/mpp-ia'
import os
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)

In [ ]:
# Remplace par l'URL de TON repo GitHub une fois créé et les fichiers src/ uploadés
REPO_URL = "https://github.com/TON_USER/mpp-ia.git"

!git clone $REPO_URL /content/mpp-ia
%cd /content/mpp-ia
!pip install -q -r requirements.txt

In [ ]:
# Symlink models/ et logs/ vers Drive pour que le checkpointing survive à une coupure de session
!rm -rf /content/mpp-ia/models /content/mpp-ia/logs
os.makedirs(f'{DRIVE_PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT_DIR}/logs', exist_ok=True)
!ln -s $DRIVE_PROJECT_DIR/models /content/mpp-ia/models
!ln -s $DRIVE_PROJECT_DIR/logs /content/mpp-ia/logs

## 2. Kaggle — téléchargement des datasets bruts

In [ ]:
# Nouveau format de token Kaggle (KGAT_...), généré depuis kaggle.com/settings -> API.
import os
os.makedirs('/root/.kaggle', exist_ok=True)

KAGGLE_API_TOKEN = "A_COMPLETER"  # <-- colle ton token KGAT_...

with open('/root/.kaggle/access_token', 'w') as f:
    f.write(KAGGLE_API_TOKEN)
!chmod 600 /root/.kaggle/access_token
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

In [ ]:
from src import config as cfg

!kaggle datasets download -d {cfg.KAGGLE_CLUB_DATASET} -p data/ --unzip
!kaggle datasets download -d {cfg.KAGGLE_INTL_DATASET} -p data/ --unzip
!ls data/

In [ ]:
# IMPORTANT : renomme les CSV téléchargés pour matcher config.py, ex :
# !mv data/nom_reel_du_fichier.csv data/club_matches_raw.csv
# !mv data/nom_reel_du_fichier2.csv data/intl_matches_raw.csv
!ls data/

## 3. Vérification des colonnes (ETAPE OBLIGATOIRE avant le build)
Si les noms affichés ci-dessous ne correspondent pas à `config.CLUB_COLUMN_MAP` / `INTL_COLUMN_MAP`,
va corriger `src/config.py` AVANT de lancer la cellule suivante.

In [ ]:
from src import data_pipeline as dp
dp.inspect_columns(cfg.RAW_CLUB_CSV)
print('---')
dp.inspect_columns(cfg.RAW_INTL_CSV)

## 4. Build dataset + split temporel

In [ ]:
df = dp.build_dataset(include_intl=True)
train_df, val_df, test_df = dp.temporal_split(df)

import numpy as np
X_train = train_df[cfg.FEATURE_COLUMNS].values.astype(np.float32)
y_train = train_df[[cfg.TARGET_HOME_GOALS, cfg.TARGET_AWAY_GOALS]].values.astype(np.float32)
X_val = val_df[cfg.FEATURE_COLUMNS].values.astype(np.float32)
y_val = val_df[[cfg.TARGET_HOME_GOALS, cfg.TARGET_AWAY_GOALS]].values.astype(np.float32)
X_test = test_df[cfg.FEATURE_COLUMNS].values.astype(np.float32)
y_test = test_df[[cfg.TARGET_HOME_GOALS, cfg.TARGET_AWAY_GOALS]].values.astype(np.float32)

input_dim = X_train.shape[1]
print('input_dim =', input_dim)

## 5. NAS génétique — LE RUN DE LA NUIT
Profil configuré dans `config.py` : pop=120, elite=12, gen=60, budget=11h.
Reprend automatiquement depuis le dernier checkpoint si la cellule est relancée après une coupure.

In [ ]:
%%javascript
// Keep-alive : simule un clic toutes les 60s pour limiter le timeout d'inactivité Colab free
// (aide, ne garantit pas 100% sur une session de 10-12h — checkpointing = vraie sécurité)
function ColabKeepAlive(){
  document.querySelector('#top-toolbar').click();
}
setInterval(ColabKeepAlive, 60000)

In [ ]:
from src import nas

best_fitness, best_config, best_state = nas.run_nas(
    X_train, y_train, X_val, y_val, input_dim, resume=True
)
print('Meilleure config trouvée :', best_config)

## 6. Entraînement final (le lendemain matin)
Relit le checkpoint NAS, prend le top-N, entraîne à fond, construit l'ensemble.

In [ ]:
from src import train as tr

ckpt = nas.load_checkpoint(f'{cfg.MODELS_DIR}/nas_checkpoint.pt')
print('Génération atteinte :', ckpt['generation'])

# Meilleure archi seule -> modèle final
state, val_nll = tr.train_final_model(X_train, y_train, X_val, y_val, input_dim, best_config)

In [ ]:
# OPTIONNEL : construire un ensemble des meilleurs candidats de la dernière génération
# (nécessite d'avoir gardé fitnesses+states de la dernière population — voir nas.py si tu veux étendre)

## 7. Backtesting sur le test set (jamais vu)

In [ ]:
from src import model as mdl
import torch

final_ckpt = torch.load(f'{cfg.MODELS_DIR}/final_model.pt')
net = tr.build_model_from_config(input_dim, final_ckpt['config'], final_ckpt['state_dict'])
net.eval()
with torch.no_grad():
    test_pred = net(torch.tensor(X_test, dtype=torch.float32))
    test_nll = mdl.poisson_nll_loss(test_pred, torch.tensor(y_test, dtype=torch.float32))
print('Test NLL (plus bas = mieux) :', test_nll.item())

# Baseline de comparaison : NLL si on prédit juste la moyenne de buts globale (sanity check)
mean_h, mean_a = y_train[:,0].mean(), y_train[:,1].mean()
baseline_pred = torch.tensor([[mean_h, mean_a]] * len(y_test), dtype=torch.float32)
baseline_nll = mdl.poisson_nll_loss(baseline_pred, torch.tensor(y_test, dtype=torch.float32))
print('Baseline (moyenne) NLL :', baseline_nll.item(), '-> ton modèle doit être en dessous')

## 8. Prédiction sur un match

In [ ]:
from src import predict

HOME_TEAM = "France"
AWAY_TEAM = "Espagne"
MATCH_DATE = "2026-07-10"

feat_vector = dp.build_feature_vector_for_match(df, HOME_TEAM, AWAY_TEAM, MATCH_DATE)
result = predict.predict_match(feat_vector, model_path=f'{cfg.MODELS_DIR}/final_model.pt')
result

## 9. Réentraînement incrémental (après chaque journée)
A relancer avec les nouveaux résultats réels une fois qu'ils sont tombés.

In [ ]:
# A relancer après chaque journée de matchs pour mettre à jour le modèle.
new_matches_df = dp.fetch_recent_results(days_back=3)

if len(new_matches_df) > 0:
    # merge avec l'historique complet pour recalculer les features avec le bon contexte
    full_hist = pd.concat([df, new_matches_df], ignore_index=True).sort_values('date')
    enriched = dp._rolling_team_features(full_hist)
    recent_feat = enriched[enriched['source'] == 'football-data-live']

    new_X = recent_feat[cfg.FEATURE_COLUMNS].fillna(df[cfg.FEATURE_COLUMNS].median()).values.astype('float32')
    new_y = recent_feat[[cfg.TARGET_HOME_GOALS, cfg.TARGET_AWAY_GOALS]].values.astype('float32')

    tr.incremental_retrain(f'{cfg.MODELS_DIR}/final_model.pt', new_X, new_y)
else:
    print('Aucun nouveau match terminé sur la période.')